In [8]:
import os, sys, json
import pandas as pd
from google.cloud import bigquery
PROJECT_ID = "project-5eb4889f-74fb-4ad5-a76"
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
elif os.environ.get("GCP_SA_KEY"):
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: project-5eb4889f-74fb-4ad5-a76


In [9]:
range_sql = f"""
SELECT MIN(business_date) AS min_date, MAX(business_date) AS max_date
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
"""

range_df = client.query(range_sql).to_dataframe()
min_date = range_df.loc[0, "min_date"]
max_date = range_df.loc[0, "max_date"]

print(min_date, max_date)

2026-09-23 2026-09-29


/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [10]:
year_min = min_date.year
year_max = max_date.year

dates = pd.date_range(f"{year_min}-01-01", f"{year_max}-12-31", freq="D")

print(len(dates))
dates[:3], dates[-3:]

365


(DatetimeIndex(['2026-01-01', '2026-01-02', '2026-01-03'], dtype='datetime64[us]', freq='D'),
 DatetimeIndex(['2026-12-29', '2026-12-30', '2026-12-31'], dtype='datetime64[us]', freq='D'))

In [11]:
dim_date = pd.DataFrame({"full_date": dates})

dim_date["date_key"]    = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"]        = dim_date["full_date"].dt.year
dim_date["quarter"]     = dim_date["full_date"].dt.quarter
dim_date["month"]       = dim_date["full_date"].dt.month
dim_date["year_month"]  = dim_date["full_date"].dt.strftime("%Y-%m")
dim_date["day_name"]    = dim_date["full_date"].dt.day_name()
dim_date["is_weekend"]  = dim_date["full_date"].dt.dayofweek >= 5

# переводимо full_date в тип date (без часу), інакше BigQuery запише як TIMESTAMP
dim_date["full_date"] = dim_date["full_date"].dt.date

dim_date = dim_date[["date_key", "full_date", "year", "quarter", "month",
                     "year_month", "day_name", "is_weekend"]]

print(dim_date.shape)
dim_date.head()

(365, 8)


,date_key,full_date,year,quarter,month,year_month,day_name,is_weekend
0,20260101,2026-01-01,2026,1,1,2026-01,Thursday,False
1,20260102,2026-01-02,2026,1,1,2026-01,Friday,False
2,20260103,2026-01-03,2026,1,1,2026-01,Saturday,True
3,20260104,2026-01-04,2026,1,1,2026-01,Sunday,True
4,20260105,2026-01-05,2026,1,1,2026-01,Monday,False


In [12]:
unknown_date = pd.DataFrame([{
    "date_key": -1,
    "full_date": pd.Timestamp("1900-01-01").date(),
    "year": -1,
    "quarter": -1,
    "month": -1,
    "year_month": "Unknown",
    "day_name": "Unknown",
    "is_weekend": False,
}])

dim_date = pd.concat([dim_date, unknown_date], ignore_index=True)
dim_date.tail()

,date_key,full_date,year,quarter,month,year_month,day_name,is_weekend
361,20261228,2026-12-28,2026,4,12,2026-12,Monday,False
362,20261229,2026-12-29,2026,4,12,2026-12,Tuesday,False
363,20261230,2026-12-30,2026,4,12,2026-12,Wednesday,False
364,20261231,2026-12-31,2026,4,12,2026-12,Thursday,False
365,-1,1900-01-01,-1,-1,-1,Unknown,Unknown,False


In [13]:
date_schema = [
    bigquery.SchemaField("date_key",    "INTEGER"),
    bigquery.SchemaField("full_date",   "DATE"),
    bigquery.SchemaField("year",        "INTEGER"),
    bigquery.SchemaField("quarter",     "INTEGER"),
    bigquery.SchemaField("month",       "INTEGER"),
    bigquery.SchemaField("year_month",  "STRING"),
    bigquery.SchemaField("day_name",    "STRING"),
    bigquery.SchemaField("is_weekend",  "BOOLEAN"),
]

DATE_TABLE_ID = f"{PROJECT_ID}.nbu_dwh.dim_date"

cfg = bigquery.LoadJobConfig(schema=date_schema, write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(dim_date, DATE_TABLE_ID, job_config=cfg).result()

print("записано рядків:", len(dim_date))

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


записано рядків: 366


In [14]:
check_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNTIF(date_key = -1) AS has_unknown
FROM `{PROJECT_ID}.nbu_dwh.dim_date`
"""
print(client.query(check_sql).to_dataframe())

# перевірка на відсутність пропусків у справжньому календарі (без Unknown)
real_dates = dim_date[dim_date["date_key"] != -1].sort_values("full_date")
gaps = pd.to_datetime(real_dates["full_date"]).diff().dropna()
print("усі кроки по 1 дню:", (gaps == pd.Timedelta(days=1)).all())

   total_rows  has_unknown
0         366            1
усі кроки по 1 дню: True


/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
